In [3]:
import pandas as pd

import shap
import matplotlib.pyplot as plt


c:\Users\jules\Desktop\Alkemists-on-AI-productivity\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
df = pd.read_csv('ai_productivity_clean.csv')
df.head()

,client,client_tier,team,task_type,seniority,task_complexity_score,brief_quality_score,deadline_pressure,scope_change_flag,pricing_model,...,profit,task_status,workflow_stage,jira_ticket,legacy_ai_flag,content_version,actual_days,margin,efficiency_ratio,rework_ratio
0,Client_F,mid,Content,report,junior,2,3.0,high,0,hourly,...,151.94,review,finalized,JIRA-49014,true,v1,5,0.305033,0.673657,0.275229
1,Client_H,low,Media,release,junior,1,2.0,medium,0,fixed,...,503.83,delivered,client_review,JIRA-84793,false,v1,2,0.594834,0.863445,0.470588
2,Client_D,low,Design,dev,junior,3,4.0,medium,0,fixed,...,1009.05,in_progress,qa,JIRA-42485,true,v2,7,0.734351,0.727811,0.320710
3,Client_E,mid,Content,design,mid,3,2.0,low,0,hourly,...,864.38,in_progress,briefing,JIRA-53111,false,v1,3,0.363321,0.854321,0.000000
4,Client_C,low,Design,article,senior,2,5.0,low,0,fixed,...,374.68,review,execution,JIRA-86006,true,v2,4,0.527755,0.748735,0.141653


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Drop ID-like columns
id_columns = ['client', 'jira_ticket', 'delivered_at', 'pricing_model', 'content_version']
df_processed = df.drop(columns=id_columns)

# Encode categorical variables
categorical_columns = df_processed.select_dtypes(include=['object', 'bool']).columns
label_encoders = {}

for col in categorical_columns:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    label_encoders[col] = le

# Display the processed dataframe
df_processed.head()

In [9]:
# Perform threshold analysis with dependence plots and SHAP values to determine the threshold for ai_usage_pct by which the margin is eroded.
# Group by seniority and task complexity.

X = df.drop(columns=['margin'])
y = df['margin']

# Train a model (example: Random Forest)
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()
model.fit(X, y)

# Explain the model predictions using SHAP
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

# Dependence plot for ai_usage_pct
shap.dependence_plot('ai_usage_pct', shap_values, X)

# Threshold analysis
def threshold_analysis(shap_values, feature, threshold):
    """
    Analyze the threshold for a feature where the margin is eroded.
    """
    feature_values = X[feature]
    erosion_points = feature_values[shap_values[:, X.columns.get_loc(feature)] < threshold]
    return erosion_points

# Example usage
threshold = -0.1  # Define a threshold for erosion
erosion_points = threshold_analysis(shap_values, 'ai_usage_pct', threshold)

# Group by seniority and task complexity
grouped = df.groupby(['seniority', 'task_complexity'])
for name, group in grouped:
    print(f"Group: {name}")
    erosion_points = threshold_analysis(shap_values, 'ai_usage_pct', threshold)
    print(erosion_points)

ValueError: could not convert string to float: 'Client_F'